In [72]:
import requests
import pandas as pd
import json
import io
from datetime import datetime

#### Test the blank endpoints

In [59]:
def test_blank_calls(endpoint):
    response = requests.get(f"http://localhost:8000/{endpoint}")
    if response.status_code == 200:
        print(f"✅ Endpoint {endpoint} is reachable with status code 200.")
        data = response.json()
        print("Response content:")
        print(json.dumps(data, indent=4)[:500], "...... (truncated)")
    else:
        print(f"❌ Endpoint {endpoint} returned status code {response.status_code}.")
        print("Response content:", response.text)

In [43]:
test_blank_calls('forecasts')
test_blank_calls('analyses-assim')
test_blank_calls('retrospectives')


✅ Endpoint forecasts is reachable with status code 200.
Response content:
{
    "api_title": "CIROH National Water Model API",
    "api_summary": "This is the version 2.0.0 of the CIROH National Water Model API with new endpoints, derived datasets, and enhanced capabilities.",
    "api_version": "2.0.0",
    "endpoint": "forecasts",
    "description": "Access the forecast configuration data for one of the three forecast types at a certain reference time against one or more reaches.",
    "immediate_data_source": {
        "long_range": "bigquery-public-data.national_w ...... (truncated)
✅ Endpoint analyses-assim is reachable with status code 200.
Response content:
{
    "api_title": "CIROH National Water Model API",
    "api_summary": "This is the version 2.0.0 of the CIROH National Water Model API with new endpoints, derived datasets, and enhanced capabilities.",
    "api_version": "2.0.0",
    "endpoint": "analyses-assim",
    "description": "Access the analysis-assimilation simulati

In [44]:
test_blank_calls('percentile-flows')
test_blank_calls('flow-metrics')
test_blank_calls('return-periods')

✅ Endpoint percentile-flows is reachable with status code 200.
Response content:
{
    "api_title": "CIROH National Water Model API",
    "api_summary": "This is the version 2.0.0 of the CIROH National Water Model API with new endpoints, derived datasets, and enhanced capabilities.",
    "api_version": "2.0.0",
    "endpoint": "percentile-flows",
    "description": "Access the streamflow magnitude data corresponding to specified or all available percentiles, derived from the daily aggregation of the NOAA National Water Model Retrospective 3.0 dataset, against one or more rea ...... (truncated)
✅ Endpoint flow-metrics is reachable with status code 200.
Response content:
{
    "api_title": "CIROH National Water Model API",
    "api_summary": "This is the version 2.0.0 of the CIROH National Water Model API with new endpoints, derived datasets, and enhanced capabilities.",
    "api_version": "2.0.0",
    "endpoint": "flow-metrics",
    "description": "Access the streamflow metrics (or indi

In [45]:
test_blank_calls('geometries')
test_blank_calls('reaches/0')

✅ Endpoint geometries is reachable with status code 200.
Response content:
{
    "api_title": "CIROH National Water Model API",
    "api_summary": "This is the version 2.0.0 of the CIROH National Water Model API with new endpoints, derived datasets, and enhanced capabilities.",
    "api_version": "2.0.0",
    "endpoint": "geometries",
    "description": "This endpoint allows users to retrieve the geometries of stream reaches.",
    "immediate_data_source": "bigquery-public-data.national_water_model.stream_network",
    "input_parameters": {
        "reach_id": {
       ...... (truncated)
✅ Endpoint reaches/0 is reachable with status code 200.
Response content:
{
    "api_title": "CIROH National Water Model API",
    "api_summary": "This is the version 2.0.0 of the CIROH National Water Model API with new endpoints, derived datasets, and enhanced capabilities.",
    "api_version": "2.0.0",
    "endpoint": "reaches/<reach_id>",
    "description": "Access a compiled set of data for a speci

#### Define the common function to request data through the API

In [201]:
API_ROOT_URL = "http://localhost:8000" # for local testing, change to actual URL when deployed
def request_to_response(endpoint, params):
    if params.get('output_format') in ['shapefile', 'geopackage']:
        print(f"⚠️ Output format {params.get('output_format')} is not directly viewable. Testing file download instead.")
        response = requests.get(f"{API_ROOT_URL}/{endpoint}", 
                            params=params, stream=True)
    else:
        response = requests.get(f"{API_ROOT_URL}/{endpoint}", 
                            params=params)
    
    if response.status_code == 200:
        print(f"✅ Endpoint {endpoint} is reachable with status code 200.")
        if params.get('output_format') == 'json':
            data = response.json()
            if params['metadata'] == True:
                metadata = data.get('metadata')
                print("User requested parameters:", metadata.get('user_request_parameters'))
                print("Data source information:", metadata.get('data_source_information'))
                print("Number of reaches:", metadata.get('number_of_reaches'))
                print("Number of timesteps:", metadata.get('number_of_timesteps'))
                print("Number of records:", metadata.get('number_of_records'))
                print("Data preview:")
                print(json.dumps(data.get('data'), indent=4)[:500], "...... (truncated)")
            else:
                print("Data preview:")
                print(json.dumps(data, indent=4)[:500], "...... (truncated)")
        if params.get('output_format') == 'csv':
            if params['metadata'] == True:
                print("Metadata preview:")
                print(response.text[:500], "...... (truncated)")  # Assuming metadata is in the first line of
                df = pd.read_csv(io.StringIO(response.text), comment='#')
            else:
                df = pd.read_csv(io.StringIO(response.text))
            print("Data preview:")
            print(df.head())
        if params.get('output_format') == 'geojson':
            data = response.json()
            if params['metadata'] == True:
                metadata = data.get('metadata')
                print("User requested parameters:", metadata.get('user_request_parameters'))
                print("Data source information:", metadata.get('data_source_information'))
                print("Number of reaches:", metadata.get('number_of_reaches'))
                print("Number of timesteps:", metadata.get('number_of_timesteps'))
                print("Number of records:", metadata.get('number_of_records'))
            print("Data preview:")
            print(json.dumps(data, indent=4)[:500], "...... (truncated)")
        if params.get('output_format') == 'shapefile':
            time_now = datetime.now().strftime("%Y%m%d%H%M%S")
            save_path = f'test_downloads/{endpoint}{time_now}.zip'
            try:
                with open(save_path, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                print(f"✅ Successfully downloaded file to: {save_path}")
            except IOError as e:
                print(f"Error saving file: {e}")
        if params.get('output_format') == 'geopackage':
            if params.get('metadata') == False:
                time_now = datetime.now().strftime("%Y%m%d%H%M%S")
                save_path = f'test_downloads/{endpoint}{time_now}.gpkg'
                try:
                    with open(save_path, 'wb') as f:
                        for chunk in response.iter_content(chunk_size=8192):
                            if chunk:
                                f.write(chunk)
                    print(f"✅ Successfully downloaded file to: {save_path}")
                except IOError as e:
                    print(f"Error saving file: {e}")
            if params['metadata'] == True:
                time_now = datetime.now().strftime("%Y%m%d%H%M%S")
                save_path = f'test_downloads/{endpoint}{time_now}.zip'
                try:
                    with open(save_path, 'wb') as f:
                        for chunk in response.iter_content(chunk_size=8192):
                            if chunk:
                                f.write(chunk)
                    print(f"✅ Successfully downloaded file to: {save_path}")
                except IOError as e:
                    print(f"Error saving file: {e}")
    else:
        print(f"❌ Endpoint {endpoint} returned status code {response.status_code}.")
        print("Response content:", response.text)

#### Test 'geometries' endpoint

In [76]:
test_geometries_endpoint = lambda params: request_to_response('geometries', params)

In [77]:
test_geometries_endpoint({
    'huc': '160202010500',
    'ordered': True,
    'metadata': True,
    'output_format': 'json'})

✅ Endpoint geometries is reachable with status code 200.
User requested parameters: huc=160202010500&ordered=True&metadata=True&output_format=json
Data source information: None
Number of reaches: 7
Number of timesteps: None
Number of records: 7
Data preview:
[
    {
        "reach_id": 10348666,
        "stream_order": 2,
        "shape_length": 0.005072048288504404,
        "geometry": "LINESTRING(-111.577360271 40.2257278710001, -111.578374871 40.2265062710001, -111.579031204 40.2273074710001, -111.579240004 40.227605071, -111.580791671 40.2290016040001, -111.581090271 40.229070404)"
    },
    {
        "reach_id": 10348680,
        "stream_order": 1,
        "shape_length": 0.025586451682171545,
        "geometry": "LINESTRING(-111.553028004 40. ...... (truncated)


In [49]:
test_geometries_endpoint({
    'huc': '160202010500',
    'ordered': True,
    'metadata': True,
    'lowest_stream_order': 2,
    'output_format': 'json'})

✅ Endpoint geometries is reachable with status code 200.
User requested parameters: huc=160202010500&ordered=True&metadata=True&lowest_stream_order=2&output_format=json
Data source information: None
Number of reaches: 2
Number of timesteps: None
Number of records: 2
Data preview:
[
    {
        "reach_id": 10348666,
        "stream_order": 2,
        "shape_length": 0.005072048288504404,
        "geometry": "LINESTRING(-111.577360271 40.2257278710001, -111.578374871 40.2265062710001, -111.579031204 40.2273074710001, -111.579240004 40.227605071, -111.580791671 40.2290016040001, -111.581090271 40.229070404)"
    },
    {
        "reach_id": 10376604,
        "stream_order": 2,
        "shape_length": 0.005896645517316316,
        "geometry": "LINESTRING(-111.581090271 40. ...... (truncated)


In [54]:
test_geometries_endpoint({
    'geom_filter': 'POLYGON ((-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -111.88137384735077 40.37069805865761, -111.88137384735077 40.40221765026246, -111.93525458166658 40.40221765026246))',
    'ordered': True,
    'metadata': True,
    'output_format': 'geojson'})

✅ Endpoint geometries is reachable with status code 200.
User requested parameters: geom_filter=POLYGON+%28%28-111.93525458166658+40.40221765026246%2C+-111.93525458166658+40.37069805865761%2C+-111.88137384735077+40.37069805865761%2C+-111.88137384735077+40.40221765026246%2C+-111.93525458166658+40.40221765026246%29%29&ordered=True&metadata=True&output_format=geojson
Data source information: None
Number of reaches: 17
Number of timesteps: None
Number of records: 17
Data preview:
{
    "type": "FeatureCollection",
    "metadata": {
        "api_title": "CIROH National Water Model API",
        "api_summary": "This is the version 2.0.0 of the CIROH National Water Model API with new endpoints, derived datasets, and enhanced capabilities.",
        "api_version": "2.0.0",
        "endpoint": "geometries",
        "description": "This endpoint allows users to retrieve the geometries of stream reaches.",
        "immediate_data_source": "bigquery-public-data.national_water_mo ...... (truncated)

In [66]:
test_geometries_endpoint({
    'reach_id': '1891586,2927567,3134443,3589508',
    'ordered': True,
    'metadata': True,
    'output_format': 'csv'})

✅ Endpoint geometries is reachable with status code 200.
Metadata preview:

# Metadata Information:
# --------------------------------------------------
# api_title: CIROH National Water Model API,
# api_summary: This is the version 2.0.0 of the CIROH National Water Model API with new endpoints, derived datasets, and enhanced capabilities.,
# api_version: 2.0.0,
# endpoint: geometries,
# description: This endpoint allows users to retrieve the geometries of stream reaches.,
# user_request_parameters: reach_id=1891586%2C2927567%2C3134443%2C3589508&ordered=True&metadata=T ...... (truncated)
Data preview:
   reach_id  stream_order  shape_length  \
0   1891586             6      0.024267   
1   2927567             8      0.013903   
2   3134443             7      0.046428   
3   3589508             4      0.034257   

                                            geometry  
0  LINESTRING(-121.336987789 37.979876274, -121.3...  
1  LINESTRING(-91.2472283019999 39.637495405, -91...  
2  LINESTR

In [70]:
test_geometries_endpoint({
    'reach_id': '1891586,2927567,3134443,3589508',
    'ordered': True,
    'metadata': False,
    'output_format': 'csv'})

✅ Endpoint geometries is reachable with status code 200.
Data preview:
   reach_id  stream_order  shape_length  \
0   1891586             6      0.024267   
1   2927567             8      0.013903   
2   3134443             7      0.046428   
3   3589508             4      0.034257   

                                            geometry  
0  LINESTRING(-121.336987789 37.979876274, -121.3...  
1  LINESTRING(-91.2472283019999 39.637495405, -91...  
2  LINESTRING(-97.9485214919999 33.8793230810001,...  
3  LINESTRING(-98.898371824 29.9637586200001, -98...  


In [80]:
test_geometries_endpoint({
    'bounding_box': '-111.705,40.160,-111.582,40.331',
    'ordered': True,
    'metadata': False,
    'output_format': 'json'})

✅ Endpoint geometries is reachable with status code 200.
Data preview:
[
    {
        "reach_id": 10328507,
        "stream_order": 1,
        "shape_length": 0.0006864000290870969,
        "geometry": "LINESTRING(-111.66476407 40.2041482710001, -111.66545047 40.2041480710001)"
    },
    {
        "reach_id": 10329053,
        "stream_order": 2,
        "shape_length": 0.05025967858868657,
        "geometry": "LINESTRING(-111.65300687 40.3645610040001, -111.652976604 40.3635770710001, -111.65315567 40.3628220040001, -111.65360387 40.361586204, -111.654321404 40.3 ...... (truncated)


In [82]:
test_geometries_endpoint({
    'gage_id': '13309220,13042500',
    'ordered': True,
    'metadata': False,
    'output_format': 'csv'})

✅ Endpoint geometries is reachable with status code 200.
Data preview:
   reach_id  stream_order  shape_length  \
0  23518785             6      0.033467   
1  24460018             6      0.051445   

                                            geometry  
0  LINESTRING(-115.017273199 44.7432525310001, -1...  
1  LINESTRING(-111.439136471 44.421713331, -111.4...  


In [83]:
test_geometries_endpoint({
    'hydroshare_id': '643dc03878704a30849536e302bdb2c0',
    'ordered': True,
    'metadata': False,
    'output_format': 'csv'})

✅ Endpoint geometries is reachable with status code 200.
Data preview:
   reach_id  stream_order  shape_length  \
0    202863             1      0.048593   
1    202875             1      0.027041   
2    202883             1      0.129878   
3    203071             1      0.145510   
4    203715             3      0.019804   

                                            geometry  
0  LINESTRING(-97.748991092 26.518197092, -97.748...  
1  LINESTRING(-97.496959026 26.5211898920001, -97...  
2  LINESTRING(-97.700409825 26.518712226, -97.664...  
3  LINESTRING(-97.858696225 26.500352892, -97.858...  
4  LINESTRING(-98.2444852249999 26.172469693, -98...  


In [84]:
test_geometries_endpoint({
    'lon': '-111.786835',
    'lat': '40.267194',
    'metadata': False,
    'output_format': 'json'})

✅ Endpoint geometries is reachable with status code 200.
Data preview:
[
    {
        "reach_id": 10329243,
        "stream_order": 2,
        "shape_length": 0.06611678049508146,
        "geometry": "LINESTRING(-111.74022127 40.264141071, -111.74407867 40.2623676040001, -111.75702087 40.261737204, -111.80569367 40.2665856040001)",
        "distance": 273.9361964276472
    }
] ...... (truncated)


In [86]:
test_geometries_endpoint({
    'geom_filter': '010600000002000000010300000001000000050000004c6c0836dbfb5bc028e032de7b3344404c6c0836dbfb5bc00450b308732f4440f67ada6d68f85bc00450b308732f4440f67ada6d68f85bc028e032de7b3344404c6c0836dbfb5bc028e032de7b33444001030000000100000005000000db2efe5e37fb5bc047651e3034324440db2efe5e37fb5bc0e5cac7b6ba30444066b8e4440cf95bc0e5cac7b6ba30444066b8e4440cf95bc047651e3034324440db2efe5e37fb5bc047651e3034324440',
    'metadata': True,
    'output_format': 'json'})

✅ Endpoint geometries is reachable with status code 200.
User requested parameters: geom_filter=010600000002000000010300000001000000050000004c6c0836dbfb5bc028e032de7b3344404c6c0836dbfb5bc00450b308732f4440f67ada6d68f85bc00450b308732f4440f67ada6d68f85bc028e032de7b3344404c6c0836dbfb5bc028e032de7b33444001030000000100000005000000db2efe5e37fb5bc047651e3034324440db2efe5e37fb5bc0e5cac7b6ba30444066b8e4440cf95bc0e5cac7b6ba30444066b8e4440cf95bc047651e3034324440db2efe5e37fb5bc047651e3034324440&metadata=True&output_format=json
Data source information: None
Number of reaches: 17
Number of timesteps: None
Number of records: 17
Data preview:
[
    {
        "reach_id": 10328245,
        "stream_order": 1,
        "shape_length": 0.0009104670591357183,
        "geometry": "LINESTRING(-111.896658603 40.4013538040001, -111.89579047 40.401079404)"
    },
    {
        "reach_id": 10328251,
        "stream_order": 1,
        "shape_length": 0.007836019243839889,
        "geometry": "LINESTRING(-111.8957904

In [87]:
test_geometries_endpoint({
    'geom_filter': 'LINESTRING (-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -111.88137384735077 40.40221765026246)',
    'metadata': True,
    'output_format': 'json'})

✅ Endpoint geometries is reachable with status code 200.
User requested parameters: geom_filter=LINESTRING+%28-111.93525458166658+40.40221765026246%2C+-111.93525458166658+40.37069805865761%2C+-111.88137384735077+40.40221765026246%29&metadata=True&output_format=json
Data source information: None
Number of reaches: 7
Number of timesteps: None
Number of records: 7
Data preview:
[
    {
        "reach_id": 10328353,
        "stream_order": 1,
        "shape_length": 0.02458858532450722,
        "geometry": "LINESTRING(-111.93022867 40.3518026040001, -111.93202767 40.356995871, -111.93229907 40.3594898710001, -111.932659003 40.3606566710001, -111.93304827 40.3612056710001, -111.933257803 40.3613658040001, -111.93418667 40.363035671, -111.93472627 40.364408204, -111.934906803 40.3654606710001, -111.934878203 40.3670166710001, -111.934670803 40.369144804, -111.93434307 40. ...... (truncated)


In [88]:
test_geometries_endpoint({
    'geom_filter': 'LINESTRING (-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -111.88137384735077 40.40221765026246)',
    'with_buffer': 200,
    'metadata': True,
    'output_format': 'json'})

✅ Endpoint geometries is reachable with status code 200.
User requested parameters: geom_filter=LINESTRING+%28-111.93525458166658+40.40221765026246%2C+-111.93525458166658+40.37069805865761%2C+-111.88137384735077+40.40221765026246%29&with_buffer=200&metadata=True&output_format=json
Data source information: None
Number of reaches: 11
Number of timesteps: None
Number of records: 11
Data preview:
[
    {
        "reach_id": 10328295,
        "stream_order": 1,
        "shape_length": 0.010686369959456446,
        "geometry": "LINESTRING(-111.93306067 40.375209404, -111.932642803 40.3763078710001, -111.932374003 40.3768114040001, -111.931746403 40.377749871, -111.92998267 40.379627204, -111.92944487 40.380679871, -111.927890803 40.382763004, -111.92756227 40.3836554710001, -111.927562803 40.3841588710001)"
    },
    {
        "reach_id": 10328353,
        "stream_order": 1,
        "shap ...... (truncated)


In [85]:
test_geometries_endpoint({
    'lon': '-111.786835',
    'lat': '40.267194',
    'with_buffer': 500,
    'metadata': True,
    'output_format': 'json'})

✅ Endpoint geometries is reachable with status code 200.
User requested parameters: lon=-111.786835&lat=40.267194&with_buffer=500&metadata=True&output_format=json
Data source information: None
Number of reaches: 1
Number of timesteps: None
Number of records: 1
Data preview:
[
    {
        "reach_id": 10329243,
        "stream_order": 2,
        "shape_length": 0.06611678049508146,
        "geometry": "LINESTRING(-111.74022127 40.264141071, -111.74407867 40.2623676040001, -111.75702087 40.261737204, -111.80569367 40.2665856040001)"
    }
] ...... (truncated)


In [79]:
test_geometries_endpoint({
    'reach_id': '1891586,2927567,3134443,3589508',
    'ordered': True,
    'metadata': False,
    'output_format': 'shapefile'})

⚠️ Output format shapefile is not directly viewable. Testing file download instead.
✅ Endpoint geometries is reachable with status code 200.
✅ Successfully downloaded file to: test_downloads/geometries20260322191645.zip


Test the 'geometries' endpoint for some invalid params

In [107]:
test_geometries_endpoint({
    'reach_id': '1891586,2927567',
    'ordered': True,
    'metadata': False,
    'output_format': 'geopackage'
})

⚠️ Output format geopackage is not directly viewable. Testing file download instead.
❌ Endpoint geometries returned status code 422.
Response content: {"detail":[{"loc":["output_format"],"msg":"Value error, output_format is not one of [['csv', 'geojson', 'json', 'shapefile']] for this endpoint.","type":"value_error"}]}


#### Test 'percentile-flows' endpoint

In [113]:
test_percentile_flows_endpoint = lambda params: request_to_response('percentile-flows', params)

In [114]:
test_percentile_flows_endpoint({
    'huc': '160202010500',
    'ordered': True,
    'metadata': True,
    'output_format': 'json'})

✅ Endpoint percentile-flows is reachable with status code 200.
User requested parameters: huc=160202010500&ordered=True&metadata=True&output_format=json
Data source information: None
Number of reaches: 7
Number of timesteps: None
Number of records: 7
Data preview:
[
    {
        "reach_id": 10348666,
        "min": 0.009999999776482582,
        "perc_2": 0.009999999776482582,
        "perc_5": 0.009999999776482582,
        "perc_10": 0.019999999552965164,
        "perc_20": 0.019999999552965164,
        "perc_25": 0.019999999552965164,
        "perc_30": 0.024583332783853013,
        "perc_50": 0.029999999329447746,
        "perc_75": 0.04999999701976776,
        "perc_90": 0.08041666479160388,
        "perc_95": 0.15902082923178856,
        "perc_99": 0 ...... (truncated)


In [115]:
test_percentile_flows_endpoint({
    'hydroshare_id': '643dc03878704a30849536e302bdb2c0',
    'ordered': True,
    'metadata': True,
    'output_format': 'geojson'})

✅ Endpoint percentile-flows is reachable with status code 200.
User requested parameters: hydroshare_id=643dc03878704a30849536e302bdb2c0&ordered=True&metadata=True&output_format=geojson
Data source information: None
Number of reaches: 10697
Number of timesteps: None
Number of records: 10697
Data preview:
{
    "type": "FeatureCollection",
    "metadata": {
        "api_title": "CIROH National Water Model API",
        "api_summary": "This is the version 2.0.0 of the CIROH National Water Model API with new endpoints, derived datasets, and enhanced capabilities.",
        "api_version": "2.0.0",
        "endpoint": "percentile-flows",
        "description": "Access the streamflow magnitude data corresponding to specified or all available percentiles, derived from the daily aggregation of the NOAA  ...... (truncated)


In [116]:
test_percentile_flows_endpoint({
    'gage_id': '13309220,13042500,13295000',
    'ordered': True,
    'metadata': True,
    'output_format': 'csv'})

✅ Endpoint percentile-flows is reachable with status code 200.
Metadata preview:

# Metadata Information:
# --------------------------------------------------
# api_title: CIROH National Water Model API,
# api_summary: This is the version 2.0.0 of the CIROH National Water Model API with new endpoints, derived datasets, and enhanced capabilities.,
# api_version: 2.0.0,
# endpoint: percentile-flows,
# description: Access the streamflow magnitude data corresponding to specified or all available percentiles, derived from the daily aggregation of the NOAA National Water Model Ret ...... (truncated)
Data preview:
   reach_id        min     perc_2     perc_5    perc_10    perc_20    perc_25  \
0  23478959   0.440000   0.633867   0.830000   1.050000   1.441000   1.632917   
1  23518785   6.025833   8.035883   9.817583  12.077666  15.487500  16.706562   
2  24460018  13.110000  13.969999  14.222229  14.639999  15.392666  15.696041   

     perc_30    perc_50    perc_75     perc_90     perc_95  

In [121]:
test_percentile_flows_endpoint({
    'lat': '40.267194',
    'lon': '-111.876',
    'ordered': True,
    'metadata': True,
    'output_format': 'json'})

✅ Endpoint percentile-flows is reachable with status code 200.
User requested parameters: lat=40.267194&lon=-111.876&ordered=True&metadata=True&output_format=json
Data source information: None
Number of reaches: 1
Number of timesteps: None
Number of records: 1
Data preview:
[
    {
        "reach_id": 10328453,
        "min": 0.0,
        "perc_2": 0.0,
        "perc_5": 0.0,
        "perc_10": 0.0,
        "perc_20": 0.0,
        "perc_25": 0.0,
        "perc_30": 0.0,
        "perc_50": 0.0,
        "perc_75": 0.0,
        "perc_90": 0.009999999776482582,
        "perc_95": 0.009999999776482582,
        "perc_99": 0.019999999552965164,
        "max": 0.22458332839111486
    }
] ...... (truncated)


In [120]:
test_percentile_flows_endpoint({
    'lat': '40.267194',
    'lon': '-111.876',
    'with_buffer': 1000,
    'ordered': True,
    'metadata': True,
    'output_format': 'json'})

✅ Endpoint percentile-flows is reachable with status code 200.
User requested parameters: lat=40.267194&lon=-111.876&with_buffer=1000&ordered=True&metadata=True&output_format=json
Data source information: None
Number of reaches: 1
Number of timesteps: None
Number of records: 1
Data preview:
[
    {
        "reach_id": 10328453,
        "min": 0.0,
        "perc_2": 0.0,
        "perc_5": 0.0,
        "perc_10": 0.0,
        "perc_20": 0.0,
        "perc_25": 0.0,
        "perc_30": 0.0,
        "perc_50": 0.0,
        "perc_75": 0.0,
        "perc_90": 0.009999999776482582,
        "perc_95": 0.009999999776482582,
        "perc_99": 0.019999999552965164,
        "max": 0.22458332839111486
    }
] ...... (truncated)


In [123]:
test_percentile_flows_endpoint({
    'geom_filter': 'MULTILINESTRING ((-111.93525458166658 40.40221765026246, -111.88137384735077 40.40221765026246, -111.88137384735077 40.37069805865761), (-111.88137384735077 40.40221765026246, -111.92525458166658 40.39221765026246, -111.92525458166658 40.38069805865761) )',
    'ordered': True,
    'metadata': True,
    'output_format': 'geojson'})

✅ Endpoint percentile-flows is reachable with status code 200.
User requested parameters: geom_filter=MULTILINESTRING+%28%28-111.93525458166658+40.40221765026246%2C+-111.88137384735077+40.40221765026246%2C+-111.88137384735077+40.37069805865761%29%2C+%28-111.88137384735077+40.40221765026246%2C+-111.92525458166658+40.39221765026246%2C+-111.92525458166658+40.38069805865761%29+%29&ordered=True&metadata=True&output_format=geojson
Data source information: None
Number of reaches: 6
Number of timesteps: None
Number of records: 6
Data preview:
{
    "type": "FeatureCollection",
    "metadata": {
        "api_title": "CIROH National Water Model API",
        "api_summary": "This is the version 2.0.0 of the CIROH National Water Model API with new endpoints, derived datasets, and enhanced capabilities.",
        "api_version": "2.0.0",
        "endpoint": "percentile-flows",
        "description": "Access the streamflow magnitude data corresponding to specified or all available percentiles, derive

In [127]:
test_percentile_flows_endpoint({
    'reach_id': '1891586',
    'ordered': True,
    'metadata': True,
    'percentiles': '5,50,95',
    'output_format': 'json'})

✅ Endpoint percentile-flows is reachable with status code 200.
User requested parameters: reach_id=1891586&ordered=True&metadata=True&percentiles=5%2C50%2C95&output_format=json
Data source information: None
Number of reaches: 1
Number of timesteps: None
Number of records: 1
Data preview:
[
    {
        "reach_id": 1891586,
        "perc_5": 2.319999933242798,
        "perc_50": 2.7325000166893005,
        "perc_95": 34.44733267625166
    }
] ...... (truncated)


In [124]:
test_percentile_flows_endpoint({
    'reach_id': '1891586,2927567,3134443,3589508',
    'ordered': True,
    'metadata': True,
    'output_format': 'shapefile'})


⚠️ Output format shapefile is not directly viewable. Testing file download instead.
✅ Endpoint percentile-flows is reachable with status code 200.
✅ Successfully downloaded file to: test_downloads/percentile-flows20260322232855.zip


#### Test 'flow-metrics' endpoint

In [125]:
test_flow_metrics_endpoint = lambda params: request_to_response('flow-metrics', params)

In [129]:
test_flow_metrics_endpoint({
    'huc': '1602020105',
    'ordered': True,
    'metadata': True,
    'lowest_stream_order': 2,
    'output_format': 'json'})

✅ Endpoint flow-metrics is reachable with status code 200.
User requested parameters: huc=1602020105&ordered=True&metadata=True&lowest_stream_order=2&output_format=json
Data source information: None
Number of reaches: 2
Number of timesteps: None
Number of records: 2
Data preview:
[
    {
        "reach_id": 10348666,
        "monthwise_mean": [
            0.03,
            0.02,
            0.03,
            0.07,
            0.14,
            0.14,
            0.07,
            0.04,
            0.04,
            0.03,
            0.03,
            0.03
        ],
        "monthwise_cov": [
            45.09,
            42.44,
            106.4,
            97.27,
            111.57,
            135.7,
            122.94,
            30.76,
            31.64,
         ...... (truncated)


In [130]:
test_flow_metrics_endpoint({
    'reach_id': '1891586',
    'ordered': True,
    'metadata': True,
    'metrics': 'half_flow_date, sevenQ10, flashiness_index, slope_fdc',
    'output_format': 'json'})

✅ Endpoint flow-metrics is reachable with status code 200.
User requested parameters: reach_id=1891586&ordered=True&metadata=True&metrics=half_flow_date%2C+sevenQ10%2C+flashiness_index%2C+slope_fdc&output_format=json
Data source information: None
Number of reaches: 1
Number of timesteps: None
Number of records: 1
Data preview:
[
    {
        "reach_id": 1891586,
        "half_flow_date": [
            83.32558139534883,
            24.125723253008605
        ],
        "sevenQ10": 2.28328391534319,
        "flashiness_index": [
            0.0539888713430496,
            0.046643847821606445,
            0.029128987195879884,
            0.029796465509195198,
            122.82580650215074
        ],
        "slope_fdc": 0.004127826209440222
    }
] ...... (truncated)


In [ ]:
test_flow_metrics_endpoint({
    'bounding_box': '-111.705,40.160,-111.582,40.331',
    'ordered': True,
    'metadata': True,
    'lowest_stream_order': 2,
    'output_format': 'json'})

✅ Endpoint flow-metrics is reachable with status code 200.
User requested parameters: bounding_box=-111.705%2C40.160%2C-111.582%2C40.331&ordered=True&metadata=True&lowest_stream_order=2&output_format=json
Data source information: None
Number of reaches: 42
Number of timesteps: None
Number of records: 42
Data preview:
[
    {
        "reach_id": 10329053,
        "monthwise_mean": [
            0.08,
            0.08,
            0.09,
            0.13,
            0.27,
            0.37,
            0.22,
            0.12,
            0.09,
            0.09,
            0.08,
            0.08
        ],
        "monthwise_cov": [
            31.6,
            35.31,
            55.48,
            66.27,
            64.08,
            77.48,
            115.98,
            69.69,
            31.79,
           ...... (truncated)


In [134]:
test_flow_metrics_endpoint({
    'geom_filter': 'POINT (-111.786835 40.267194)',
    'ordered': True,
    'metadata': True,
    'lowest_stream_order': 2,
    'output_format': 'shapefile'})

⚠️ Output format shapefile is not directly viewable. Testing file download instead.
✅ Endpoint flow-metrics is reachable with status code 200.
✅ Successfully downloaded file to: test_downloads/flow-metrics20260322233918.zip


#### Test 'return-periods' endpoint

In [137]:
test_return_periods = lambda params: request_to_response('return-periods', params)

In [138]:
test_return_periods({
    'huc': '160202010500',
    'metadata': True,
    'output_format': 'json'})

✅ Endpoint return-periods is reachable with status code 200.
User requested parameters: huc=160202010500&metadata=True&output_format=json
Data source information: None
Number of reaches: 7
Number of timesteps: None
Number of records: 7
Data preview:
[
    {
        "reach_id": 10376202,
        "return_period_2": 0.01,
        "return_period_5": 0.01,
        "return_period_10": 0.01,
        "return_period_25": 0.01,
        "return_period_50": 0.01,
        "return_period_100": 0.01
    },
    {
        "reach_id": 10348666,
        "return_period_2": 0.7,
        "return_period_5": 1.41,
        "return_period_10": 1.88,
        "return_period_25": 2.48,
        "return_period_50": 2.92,
        "return_period_100": 3.36
    },
    {
    ...... (truncated)


In [140]:
test_return_periods({
    'geom_filter': 'LINESTRING (-111.93525458166658 40.40221765026246, -111.88137384735077 40.40221765026246, -111.88137384735077 40.37069805865761)',
    'return_periods': '10,50,100',
    'metadata': True,
    'output_format': 'json'})

✅ Endpoint return-periods is reachable with status code 200.
User requested parameters: geom_filter=LINESTRING+%28-111.93525458166658+40.40221765026246%2C+-111.88137384735077+40.40221765026246%2C+-111.88137384735077+40.37069805865761%29&return_periods=10%2C50%2C100&metadata=True&output_format=json
Data source information: None
Number of reaches: 4
Number of timesteps: None
Number of records: 4
Data preview:
[
    {
        "reach_id": 10328305,
        "return_period_10": 7.79,
        "return_period_50": 12.69,
        "return_period_100": 14.76
    },
    {
        "reach_id": 10328235,
        "return_period_10": 103.0,
        "return_period_50": 117.18,
        "return_period_100": 123.18
    },
    {
        "reach_id": 10329069,
        "return_period_10": 0.04,
        "return_period_50": 0.06,
        "return_period_100": 0.07
    },
    {
        "reach_id": 10328285,
        "return_peri ...... (truncated)


In [ ]:
test_return_periods({
    'gage_id': '13309220',
    'return_periods': '100',
    'metadata': False,
    'output_format': 'geojson'})

✅ Endpoint return-periods is reachable with status code 200.
Data preview:
{
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "id": 23518785,
            "geometry": {
                "type": "LineString",
                "coordinates": [
                    [
                        -115.017273,
                        44.743253
                    ],
                    [
                        -115.017273,
                        44.743074
                    ],
                    [
                        -11 ...... (truncated)


In [144]:
test_return_periods({
    'gage_id': '13309220, 13042500, 13295000',
    'return_periods': '100',
    'metadata': False,
    'output_format': 'shapefile'})

⚠️ Output format shapefile is not directly viewable. Testing file download instead.
✅ Endpoint return-periods is reachable with status code 200.
✅ Successfully downloaded file to: test_downloads/return-periods20260322235231.zip


In [146]:
test_return_periods({
    'reach_id': '1891586,2927567,3134443,3589508',
    'return_periods': '100',
    'metadata': False,
    'output_format': 'csv'})

✅ Endpoint return-periods is reachable with status code 200.
Data preview:
   reach_id  return_period_100
0   1891586             428.36
1   2927567           18104.32
2   3589508            1749.95
3   3134443            3206.19


#### Test 'retrospectives' endpoint

In [145]:
test_retrospectives_endpoint = lambda params: request_to_response('retrospectives', params)

In [150]:
test_retrospectives_endpoint({
    'reach_id': '1891586',
    'ordered': True,
    'metadata': True,
    'output_format': 'json'})

✅ Endpoint retrospectives is reachable with status code 200.
User requested parameters: reach_id=1891586&ordered=True&metadata=True&output_format=json
Data source information: None
Number of reaches: 1
Number of timesteps: 745
Number of records: 745
Data preview:
[
    {
        "reach_id": 1891586,
        "time": "2023-01-23T22:00:00+0000",
        "streamflow": 93.43999481201172,
        "velocity": 0.20999999344348907
    },
    {
        "reach_id": 1891586,
        "time": "2023-01-27T00:00:00+0000",
        "streamflow": 63.63999938964844,
        "velocity": 0.17999999225139618
    },
    {
        "reach_id": 1891586,
        "time": "2023-01-26T06:00:00+0000",
        "streamflow": 69.05999755859375,
        "velocity": 0.1899999976158142
    } ...... (truncated)


In [151]:
test_retrospectives_endpoint({
    'reach_id': '2927567,3134443',
    'start_time': '2020-07-01T00:00:00',
    'end_time': '2020-07-07T08:00:00',
    'ordered': True,
    'metadata': True,
    'output_format': 'json'})

✅ Endpoint retrospectives is reachable with status code 200.
User requested parameters: reach_id=2927567%2C3134443&start_time=2020-07-01T00%3A00%3A00&end_time=2020-07-07T08%3A00%3A00&ordered=True&metadata=True&output_format=json
Data source information: None
Number of reaches: 2
Number of timesteps: 153
Number of records: 306
Data preview:
[
    {
        "reach_id": 3134443,
        "time": "2020-07-05T05:00:00+0000",
        "streamflow": 64.18000030517578,
        "velocity": 0.17000000178813934
    },
    {
        "reach_id": 3134443,
        "time": "2020-07-06T23:00:00+0000",
        "streamflow": 61.77000045776367,
        "velocity": 0.17000000178813934
    },
    {
        "reach_id": 3134443,
        "time": "2020-07-05T13:00:00+0000",
        "streamflow": 63.349998474121094,
        "velocity": 0.17000000178813934
    ...... (truncated)


In [174]:
test_retrospectives_endpoint({
    'huc': '160202010500',
    'start_time': '2020-07-01T00:00:00',
    'end_time': '2020-07-07T08:00:00',
    'ordered': True,
    'metadata': False,
    'output_format': 'geopackage'})

⚠️ Output format geopackage is not directly viewable. Testing file download instead.
✅ Endpoint retrospectives is reachable with status code 200.
✅ Successfully downloaded file to: test_downloads/retrospectives20260323013613.gpkg


In [202]:
test_retrospectives_endpoint({
    'huc': '160202010500',
    'start_time': '2020-07-01T00:00:00',
    'end_time': '2020-07-07T08:00:00',
    'ordered': True,
    'metadata': True,
    'output_format': 'geopackage'})

⚠️ Output format geopackage is not directly viewable. Testing file download instead.
✅ Endpoint retrospectives is reachable with status code 200.
✅ Successfully downloaded file to: test_downloads/retrospectives20260323025609.zip


In [203]:
test_retrospectives_endpoint({
    'huc': '160202010500',
    'start_time': '2020-07-01T00:00:00',
    'end_time': '2020-07-07T08:00:00',
    'ordered': True,
    'metadata': False,
    'output_format': 'geopackage'})

⚠️ Output format geopackage is not directly viewable. Testing file download instead.
✅ Endpoint retrospectives is reachable with status code 200.
✅ Successfully downloaded file to: test_downloads/retrospectives20260323025627.gpkg


#### Test 'forecasts' endpoint

In [90]:
test_forecasts_endpoint = lambda params: request_to_response('forecasts', params)

In [97]:
test_forecasts_endpoint({
    'huc': '160202010500',
    'ordered': True,
    'metadata': True,
    'output_format': 'json'})

✅ Endpoint forecasts is reachable with status code 200.
User requested parameters: huc=160202010500&ordered=True&metadata=True&output_format=json
Data source information: None
Number of reaches: 4
Number of timesteps: 18
Number of records: 72
Data preview:
[
    {
        "reach_id": 10348692,
        "reference_time": "2026-01-25T23:00:00+0000",
        "time": "2026-01-26T00:00:00+0000",
        "ensemble": "average",
        "streamflow": 0.0,
        "velocity": 0.22999998927116394
    },
    {
        "reach_id": 10348692,
        "reference_time": "2026-01-25T23:00:00+0000",
        "time": "2026-01-26T01:00:00+0000",
        "ensemble": "average",
        "streamflow": 0.0,
        "velocity": 0.22999998927116394
    },
    {
        "reach ...... (truncated)


In [101]:
test_forecasts_endpoint({
    'reach_id': '1891586,2927567,3134443',
    'reference_time': '2026-01-01T00:00:00',
    'time_zone': 'US/Mountain',
    'metadata': True,
    'output_format': 'geojson'})

✅ Endpoint forecasts is reachable with status code 200.
User requested parameters: reach_id=1891586%2C2927567%2C3134443&reference_time=2026-01-01T00%3A00%3A00&time_zone=US%2FMountain&metadata=True&output_format=geojson
Data source information: None
Number of reaches: 3
Number of timesteps: 18
Number of records: 54
Data preview:
{
    "type": "FeatureCollection",
    "metadata": {
        "api_title": "CIROH National Water Model API",
        "api_summary": "This is the version 2.0.0 of the CIROH National Water Model API with new endpoints, derived datasets, and enhanced capabilities.",
        "api_version": "2.0.0",
        "endpoint": "forecasts",
        "description": "Access the forecast configuration data for one of the three forecast types at a certain reference time against one or more reaches.",
        "immed ...... (truncated)


In [110]:
test_forecasts_endpoint({
    'gage_id': '13309220',
    'reference_time': '2026-01-03T00:00:00',
    'time_zone': 'America/Boise',
    'metadata': True,
    'output_format': 'json'})

✅ Endpoint forecasts is reachable with status code 200.
User requested parameters: gage_id=13309220&reference_time=2026-01-03T00%3A00%3A00&time_zone=America%2FBoise&metadata=True&output_format=json
Data source information: None
Number of reaches: 1
Number of timesteps: 18
Number of records: 18
Data preview:
[
    {
        "reach_id": 23518785,
        "reference_time": "2026-01-03T00:00:00-0700",
        "time": "2026-01-03T04:00:00-0700",
        "ensemble": "average",
        "streamflow": 27.189998626708984,
        "velocity": 0.7999999523162842
    },
    {
        "reach_id": 23518785,
        "reference_time": "2026-01-03T00:00:00-0700",
        "time": "2026-01-03T02:00:00-0700",
        "ensemble": "average",
        "streamflow": 28.170000076293945,
        "velocity": 0.8100000023841858 ...... (truncated)


In [ ]:
test_forecasts_endpoint({
    'forecast_type': 'short_range',
    'gage_id': '13309220',
    'reference_time': '2026-01-03T00:00:00',
    'metadata': True,
    'output_format': 'json'})

In [136]:
test_forecasts_endpoint({
    'forecast_type': 'medium_range',
    'geom_filter': 'POINT(-111.77 40.758)',
    'with_buffer': 1000.0,
    'reference_time': '2022-10-01T00:00:00',
    'metadata': True,
    'output_format': 'json'})

✅ Endpoint forecasts is reachable with status code 200.
User requested parameters: forecast_type=medium_range&geom_filter=POINT%28-111.77+40.758%29&with_buffer=1000.0&reference_time=2022-10-01T00%3A00%3A00&metadata=True&output_format=json
Data source information: None
Number of reaches: 4
Number of timesteps: 240
Number of records: 960
Data preview:
[
    {
        "reach_id": 10390244,
        "reference_time": "2022-10-01T00:00:00+0000",
        "time": "2022-10-03T15:00:00+0000",
        "ensemble": "average",
        "streamflow": 0.0,
        "velocity": 0.029999999329447746
    },
    {
        "reach_id": 10389426,
        "reference_time": "2022-10-01T00:00:00+0000",
        "time": "2022-10-08T19:00:00+0000",
        "ensemble": "average",
        "streamflow": 0.0,
        "velocity": 0.009999999776482582
    },
    {
        "rea ...... (truncated)


In [204]:
test_forecasts_endpoint({
    'huc': '160202010500',
    'forecast_type': 'short_range',
    'reference_time': '2020-07-01T00:00:00',
    'ordered': True,
    'metadata': True,
    'output_format': 'geopackage'})

⚠️ Output format geopackage is not directly viewable. Testing file download instead.
✅ Endpoint forecasts is reachable with status code 200.
✅ Successfully downloaded file to: test_downloads/forecasts20260323025647.zip


#### Test 'analyses-assim' endpoint

In [179]:
test_analyses_assim_endpoint = lambda params: request_to_response('analyses-assim', params)

In [180]:
test_analyses_assim_endpoint({
    'reach_id': '3134443',
    'start_time': '2020-07-01T00:00:00',
    'end_time': '2020-07-07T08:00:00',
    'ordered': True,
    'metadata': True,
    'output_format': 'json'})

✅ Endpoint analyses-assim is reachable with status code 200.
User requested parameters: reach_id=3134443&start_time=2020-07-01T00%3A00%3A00&end_time=2020-07-07T08%3A00%3A00&ordered=True&metadata=True&output_format=json
Data source information: None
Number of reaches: 1
Number of timesteps: 153
Number of records: 153
Data preview:
[
    {
        "reach_id": 3134443,
        "time": "2020-07-01T00:00:00+0000",
        "streamflow": 37.380001068115234,
        "velocity": 0.1599999964237213
    },
    {
        "reach_id": 3134443,
        "time": "2020-07-01T01:00:00+0000",
        "streamflow": 37.65999984741211,
        "velocity": 0.1599999964237213
    },
    {
        "reach_id": 3134443,
        "time": "2020-07-01T02:00:00+0000",
        "streamflow": 36.529998779296875,
        "velocity": 0.1599999964237213
    } ...... (truncated)


In [184]:
test_analyses_assim_endpoint({
    'reach_id': '23518785',
    'start_time': '2026-01-24T00:00:00',
    'end_time': '2026-01-25T00:00:00',
    'ordered': True,
    'metadata': True,
    'output_format': 'geojson'})

✅ Endpoint analyses-assim is reachable with status code 200.
User requested parameters: reach_id=23518785&start_time=2026-01-24T00%3A00%3A00&end_time=2026-01-25T00%3A00%3A00&ordered=True&metadata=True&output_format=geojson
Data source information: None
Number of reaches: 1
Number of timesteps: 25
Number of records: 25
Data preview:
{
    "type": "FeatureCollection",
    "metadata": {
        "api_title": "CIROH National Water Model API",
        "api_summary": "This is the version 2.0.0 of the CIROH National Water Model API with new endpoints, derived datasets, and enhanced capabilities.",
        "api_version": "2.0.0",
        "endpoint": "analyses-assim",
        "description": "Access the analysis-assimilation simulation data for one or more of its offset runs against one or more reaches.",
        "immediate_data_sour ...... (truncated)


#### Test 'reaches/<reach_id>' endpoint

In [197]:
def test_reaches_endpoint(reach_id, params):
    response = requests.get(f"{API_ROOT_URL}/reaches/{reach_id}", 
                              params=params)
    if response.status_code == 200:
        print(f"✅ Endpoint reaches is reachable with status code 200.")
        data = response.json()
        if params['metadata'] == True:
            metadata = data.get('metadata')
            print("User requested parameters:", metadata.get('user_request_parameters'))
            print("Data source information:", metadata.get('data_source_information'))
            print("Number of reaches:", metadata.get('number_of_reaches'))
            print("Number of timesteps:", metadata.get('number_of_timesteps'))
            print("Number of records:", metadata.get('number_of_records'))
        print("Data preview:")
        print(json.dumps(data, indent=4)[:500], "...... (truncated)")
    else:
        print(f"❌ Endpoint 'reaches' returned status code {response.status_code}.")
        print("Response content:", response.text)

In [208]:
test_reaches_endpoint(reach_id = '3134443',
    params = {'metadata': True})

✅ Endpoint reaches is reachable with status code 200.
User requested parameters: metadata=True
Data source information: None
Number of reaches: 1
Number of timesteps: {'reach_id': 1, 'flow_metrics': 1, 'return_periods': 1, 'analyses_assim': 723, 'forecasts_short_range': 18, 'forecasts_medium_range': 1260, 'forecasts_long_range': 480}
Number of records: {'reach_id': 1, 'flow_metrics': 1, 'return_periods': 1, 'analyses_assim': 723, 'forecasts_short_range': 18, 'forecasts_medium_range': 1260, 'forecasts_long_range': 480}
Data preview:
{
    "type": "Feature",
    "geometry": "LINESTRING(-97.9485214919999 33.8793230810001, -97.9462498249999 33.8796198140001, -97.943985292 33.880206881, -97.941630092 33.880311014, -97.938205625 33.880077881, -97.9364204249999 33.879397281, -97.933719025 33.8782152140001, -97.9310010249999 33.8763552140001, -97.9290774249999 33.874837214, -97.926850825 33.873776214, -97.9247084249999 33.8729400140001, -97.923638425 33.8725702140001, -97.9182020919999 33.8703

In [209]:
test_reaches_endpoint(reach_id = '3134443',
    params = {'reference_time': '2020-07-01T00:00:00',
        'metadata': True})

✅ Endpoint reaches is reachable with status code 200.
User requested parameters: reference_time=2020-07-01T00%3A00%3A00&metadata=True
Data source information: None
Number of reaches: 1
Number of timesteps: {'reach_id': 1, 'flow_metrics': 1, 'return_periods': 1, 'analyses_assim': 2160, 'forecasts_short_range': 18, 'forecasts_medium_range': 488, 'forecasts_long_range': 480}
Number of records: {'reach_id': 1, 'flow_metrics': 1, 'return_periods': 1, 'analyses_assim': 2160, 'forecasts_short_range': 18, 'forecasts_medium_range': 488, 'forecasts_long_range': 480}
Data preview:
{
    "type": "Feature",
    "geometry": "LINESTRING(-97.9485214919999 33.8793230810001, -97.9462498249999 33.8796198140001, -97.943985292 33.880206881, -97.941630092 33.880311014, -97.938205625 33.880077881, -97.9364204249999 33.879397281, -97.933719025 33.8782152140001, -97.9310010249999 33.8763552140001, -97.9290774249999 33.874837214, -97.926850825 33.873776214, -97.9247084249999 33.8729400140001, -97.923638425 33.8